# COE 311K Midterm Project: U.S. GDP Growth Rate Analysis

## Introduction
In this project, I'm analyzing the U.S. quarterly GDP growth rates from 2010 to 2023. I got the dataset from the U.S. Bureau of Economic Analysis (BEA). The goal here is to apply three different methods to fit a curve to this data: Least Squares Linear Regression, a degree-4 Polynomial Fit, and Natural Cubic Spline Interpolation. 

I wrote all the core algorithms from scratch (like solving the Thomas algorithm and normal equations) without using built-in curve-fitting functions like `numpy.polyfit` or `scipy.interpolate`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import csv

# Load the GDP dataset
dates = []
full_y = []

# This is the subset of quarters given in the assignment table
subset_quarters = [
    ('2010', 'Q1'), ('2011', 'Q1'), ('2012', 'Q1'), ('2013', 'Q1'),
    ('2014', 'Q1'), ('2014', 'Q3'), ('2015', 'Q2'), ('2016', 'Q1'),
    ('2016', 'Q3'), ('2016', 'Q4'), ('2017', 'Q1'), ('2018', 'Q1'),
    ('2019', 'Q1'), ('2020', 'Q1'), ('2020', 'Q2'), ('2020', 'Q3'),
    ('2021', 'Q1'), ('2022', 'Q1'), ('2023', 'Q2'), ('2023', 'Q4')
]

# Read the CSV to get the full timeline
with open('gdp_data.csv', 'r') as f:
    reader = csv.reader(f)
    header = next(reader)
    for row in reader:
        dates.append(row[0])
        full_y.append(float(row[1]))

start_idx = dates.index('2010-01-01')
end_idx = dates.index('2023-10-01')

# Set up the full x and y values for the 56 quarters
x_full = np.arange(1, 57)
y_full = np.array(full_y[start_idx:end_idx+1])
date_full = dates[start_idx:end_idx+1]

# Map the subset back to standard quarter indices (x = 1 to 56)
month_map = {'Q1': '01-01', 'Q2': '04-01', 'Q3': '07-01', 'Q4': '10-01'}
x_subset = []
y_subset = []

for year, q in subset_quarters:
    date_str = f"{year}-{month_map[q]}"
    idx = date_full.index(date_str)
    x_subset.append(idx + 1)
    y_subset.append(y_full[idx])

x_subset = np.array(x_subset, dtype=float)
y_subset = np.array(y_subset, dtype=float)



## Part A — Cubic Spline Interpolation

### 1. System Set up
To build a natural cubic spline through $n$ data points, we connect them with piecewise cubic polynomials $S_i(x)$. 

Because there are $n-1$ segments and each cubic equation has 4 coefficients, there are $4(n-1)$ total unknowns. To solve for them, we set up conditions:
- **0th, 1st, and 2nd Derivative Continuity:** The curves must touch exactly at each interior knot, and their slopes and concavities must match. This gives continuous $C^0$, $C^1$, and $C^2$.
- **Boundary Conditions:** A "natural" spline means the second derivative at the very first and very last point is zero ($S''(x_1)=0$ and $S''(x_n)=0$).

This reduces down into a smaller tridiagonal system of just $n-2$ equations where the unknowns are the second derivatives ($c$) at the interior points. I implemented the Thomas Algorithm to solve this efficiently.


In [ ]:
def thomas_algorithm(a, b, c, d):
    # Solves a tridiagonal matrix system, O(n) complexity.
    n = len(d)
    c_prime = np.zeros(n-1)
    d_prime = np.zeros(n)
    x = np.zeros(n)
    
    # Forward sweep
    c_prime[0] = c[0] / b[0]
    d_prime[0] = d[0] / b[0]
    for i in range(1, n-1):
        m = b[i] - a[i-1] * c_prime[i-1]
        c_prime[i] = c[i] / m
        d_prime[i] = (d[i] - a[i-1] * d_prime[i-1]) / m
        
    m = b[n-1] - a[n-2] * c_prime[n-2]
    d_prime[n-1] = (d[n-1] - a[n-2] * d_prime[n-2]) / m
    
    # Back substitution
    x[n-1] = d_prime[n-1]
    for i in range(n-2, -1, -1):
        x[i] = d_prime[i] - c_prime[i] * x[i+1]
    return x

def natural_cubic_spline(x, y):
    n = len(x)
    h = np.diff(x)
    
    # Set up the tridiagonal matrix for the second derivatives
    num_eqs = n - 2
    A_lower = h[1:num_eqs]
    A_main = 2 * (h[:-1] + h[1:])
    A_upper = h[1:num_eqs]
    
    RHS = np.zeros(num_eqs)
    for i in range(1, n-1):
        RHS[i-1] = (3 / h[i]) * (y[i+1] - y[i]) - (3 / h[i-1]) * (y[i] - y[i-1])
        
    # Solve using our custom thomas solver
    c_interior = thomas_algorithm(A_lower, A_main, A_upper, RHS)
    
    c = np.zeros(n)
    c[1:-1] = c_interior # Boundary condition c_0=0 and c_n=0 for natural spline
    
    # Solve for remaining coefficients a, b, d
    a = y[:-1]
    b = np.zeros(n-1)
    d = np.zeros(n-1)
    for i in range(n-1):
        b[i] = (y[i+1] - y[i]) / h[i] - h[i] * (2*c[i] + c[i+1]) / 3
        d[i] = (c[i+1] - c[i]) / (3 * h[i])
        
    return a, b, c[:-1], d

def evaluate_spline(x_eval, x_knots, coeffs):
    a, b, c, d = coeffs
    y_eval = np.zeros_like(x_eval)
    for idx, xi in enumerate(x_eval):
        if xi <= x_knots[0]:
            i = 0
        elif xi >= x_knots[-1]:
            i = len(x_knots) - 2
        else:
            i = np.where(x_knots <= xi)[0][-1]
            if i == len(x_knots) - 1:
                i -= 1
        
        dx = xi - x_knots[i]
        y_eval[idx] = a[i] + b[i]*dx + c[i]*dx**2 + d[i]*dx**3
    return y_eval

# 2. Evaluate at every quarter in the full range
coeffs = natural_cubic_spline(x_subset, y_subset)
y_spline_eval = evaluate_spline(x_full, x_subset, coeffs)

plt.figure(figsize=(10, 6))
plt.scatter(x_subset, y_subset, color='red', label='Constraint Subset Points', zorder=5)
plt.plot(x_full, y_spline_eval, 'b-', label='Natural Cubic Spline')
plt.title("U.S. GDP Growth: Cubic Spline Interpolation")
plt.xlabel("Quarters since 2010 Q1")
plt.ylabel("GDP Growth Rate (%)")
plt.legend()
plt.grid(True)
plt.show()



### 3. Spline Evaluation and the Runge Phenomenon

The interpolant is visibly smooth throughout most of the graph, but it goes completely crazy around the massive 2020 COVID shock outlier (-28%). 

Because the spline has to pass **exactly** through that -28% point and then sharply launch up to 34% the next quarter, while maintaining $C^2$ continuity (smooth curves), it forces the line to wildly overshoot the normal values before and after 2020. This oscillation is a classic example of the **Runge phenomenon**. 

When a dataset has extreme outliers like this, exact interpolation is a bad idea because it sacrifices numerical stability and makes the graph look unrealistic everywhere else. To fix this, it would be much better to use a **smoothing spline** (which relaxes the requirement to touch every single point in favor of a smoother curve) or **weighted least squares** to ignore the crazy COVID data.


## Part B — Polynomial & Least Squares Comparison

### 1. Degree-4 Polynomial Fit vs. Spline

Here, I'm fitting a degree-4 polynomial to the data using the least squares normal equations: $A^T A x = A^T b$. I scaled the x-values to prevent the condition number of the Vandermonde matrix from getting too high.


In [ ]:
# Set up the Vandiermonde matrix for a degree 4 poly
degree = 4

# Normalizing x to keep the condition number reasonable
x_subset_norm = (x_subset - np.mean(x_subset)) / np.std(x_subset)
A_poly_norm = np.vander(x_subset_norm, degree + 1, increasing=True)
cond_A_norm = np.linalg.cond(A_poly_norm.T @ A_poly_norm)
print(f"Condition number of A^T A (normalized x): {cond_A_norm:.2e}")

# Solve the normal equations
b_poly = A_poly_norm.T @ y_subset
p_coeffs = np.linalg.solve(A_poly_norm.T @ A_poly_norm, b_poly)

def evaluate_polynomial(x_vals, p_coeffs):
    y_vals = np.zeros_like(x_vals)
    for j, c in enumerate(p_coeffs):
        y_vals += c * (x_vals**j)
    return y_vals

x_full_norm = (x_full - np.mean(x_subset)) / np.std(x_subset)
y_poly_eval = evaluate_polynomial(x_full_norm, p_coeffs)

plt.figure(figsize=(10, 6))
plt.scatter(x_subset, y_subset, color='red', label='Data Points', zorder=5)
plt.plot(x_full, y_spline_eval, 'b--', alpha=0.5, label='Cubic Spline (Interpolation)')
plt.plot(x_full, y_poly_eval, 'g-', linewidth=2, label='Degree-4 Polynomial (Approximation)')
plt.title("Spline vs. Polynomial Fit")
plt.xlabel("Quarters since 2010 Q1")
plt.ylabel("GDP Growth Rate (%)")
plt.legend()
plt.grid(True)
plt.show()

# Calculate and plot residuals
y_poly_fit_subset = evaluate_polynomial(x_subset_norm, p_coeffs)
residuals = y_subset - y_poly_fit_subset

plt.figure(figsize=(10, 4))
plt.scatter(x_subset, residuals, color='purple')
plt.axhline(0, color='black', linestyle='--')
plt.title("Residuals: Degree-4 Polynomial Fit")
plt.xlabel("Quarters")
plt.ylabel("Residual (Actual - Fitted)")
plt.grid(True)
plt.show()



**Approximation vs Interpolation Trade-off:**
Looking at the plot, the **Degree-4 Polynomial** definitely captures the overall long-term trend better. Because it's an approximation, it sweeps cleanly through the middle of the noisy data and handles the massive 2020 COVID shock without breaking a sweat. On the flip side, the **Cubic Spline** perfectly reproduces the specific data points (interpolation), but it fails to capture the overall trend because the local outliers force the line to oscillate dramatically. 

### 2. Least Squares Linear Model (Excluding COVID)

Now, I'm fitting a strict linear trend, but filtering out the COVID quarters (indices 41 to 45).


In [ ]:
# Filter out COVID timeframe (early 2020 to early 2021)
mask = (x_subset < 41) | (x_subset > 45)
x_clean = x_subset[mask]
y_clean = y_subset[mask]

# Fit linear model (degree 1)
A_lin = np.vstack([np.ones_like(x_clean), x_clean]).T
coeffs_lin = np.linalg.solve(A_lin.T @ A_lin, A_lin.T @ y_clean)
slope = coeffs_lin[1]

print(f"Calculated Linear Slope: {slope:.4f}% per quarter")

# Plot the linear model
y_lin_eval = coeffs_lin[0] + coeffs_lin[1] * x_full

plt.figure(figsize=(10, 5))
plt.scatter(x_clean, y_clean, color='green', label='Normal Quarters')
plt.scatter(x_subset[~mask], y_subset[~mask], color='red', label='Excluded COVID Quarters')
plt.plot(x_full, y_lin_eval, 'k-', label='Linear Trend (Cleaned)')
plt.title("Least Squares Linear Model (Excluding COVID)")
plt.xlabel("Quarters")
plt.ylabel("GDP Growth Rate (%)")
plt.legend()
plt.grid(True)
plt.show()

# Linear Residual Plot
y_lin_clean_fit = coeffs_lin[0] + coeffs_lin[1] * x_clean
resid_lin = y_clean - y_lin_clean_fit

plt.figure(figsize=(10, 4))
plt.scatter(x_clean, resid_lin, color='purple')
plt.axhline(0, color='black', linestyle='--')
plt.title("Residuals: Linear Model (Cleaned)")
plt.xlabel("Quarters")
plt.ylabel("Residual")
plt.grid(True)
plt.show()



**Does a linear trend make economic sense?**
Honestly, not really. The calculated slope is barely trending upwards. Assuming a linear trend for GDP growth means you're assuming the economy just grows at a perfectly constant increasing rate year over year without cyclical changes. Real economies go through natural recessions and booms. A strict linear model basically ignores the business cycle entirely, so it's a bit too simple to be used for something highly complex and variable like quarterly GDP.

## Part C — Method Justification

### Recommendation
If a policymaker was trying to estimate what the GDP growth was for a missing quarter halfway between two known data points, I would definitely recommend using the **Polynomial Fit**.

Even though interpolation is technically designed to fill in missing gaps, economic data is super volatile and full of noise/outliers. Like we saw with the COVID drop, if they used a Cubic Spline, they might accidentally estimate a massive artificial spike in 2019 just because the math forced the spline to oscillate before the 2020 drop. The polynomial naturally smooths over local volatility, leading to much safer estimates of the overall trajectory of the economy.

## Part C Addendum: Big O Analysis

Big O Notation is a way to describe how much time or memory an algorithm takes as the size of the data/input ($n$) gets bigger. It helps us see the worst-case scenario.

1. **Spline Thomas Algorithm**: Normally, solving our tridiagonal matrix using regular Gaussian Elimination takes $\mathcal{O}(n^3)$ operations. But since we use the Thomas Algorithm, which is optimized for tridiagonal systems, it only takes one forward sweep and one backward substitution. This drops the complexity all the way down to **$\mathcal{O}(n)$**.
2. **Polynomial Least Squares**: Building the Vandermonde matrix and multiplying $A^T A$ takes about $\mathcal{O}(m d^2)$ time ($m$ is points, $d$ is the degree 4). Solving the normal equations for a degree-$d$ system takes $\mathcal{O}(d^3)$. So overall it's roughly **$\mathcal{O}(m d^2 + d^3)$**. Since $d=4$ is a tiny constant here, the time basically scales linearly with the data points on the scale of $\mathcal{O}(m)$.

**Does this change my recommendation?**
Nope. Both algorithms are extremely fast (scaling basically linearly) and since we are only evaluating GDP over 50ish points, neither will ever bottleneck a modern CPU anyway. Accuracy and handling outliers safely is way more important than algorithm speed here.

## Conclusion
To conclude, this project was really helpful in showing the trade-offs of curve fitting. The Degree-4 Least Squares polynomial was great for seeing the large-scale economic trends and absorbing the COVID shock, acting as a great abstract approximation. On the other hand, the Natural Cubic Spline was mathematically perfect at connecting points locally, but because it tries so hard to maintain smooth derivatives ($C^2$ continuity), it creates aggressive oscillations (the Runge phenomenon) around big outliers, making it a worse choice here for real-world economic data.
